# ColonSightAI — Google Colab Inference Microservice
This notebook runs the **EfficientNetB0 Colon Cancer Detection Model** and **Grad-CAM explainability engine** inside Google Colab (Linux environment).

### Why this solves Windows Smart App Control:
- Windows 11 Smart App Control (SAC) blocks unsigned native Python wheels (like TensorFlow `_pywrap_utils.pyd`) on Windows.
- Google Colab runs native Linux (Ubuntu with GPU/TPU acceleration) where TensorFlow executes at full speed without Windows Code Integrity restrictions.
- Your local Windows Flask app sends uploaded images here, receives predictions + Grad-CAM heatmaps, and renders the web UI and PDF reports seamlessly.

### Step 1: Install Required Dependencies

In [ ]:
!pip install flask pyngrok tensorflow pillow opencv-python-headless

### Step 2: Upload `efficientnetb0_colon.h5`
Upload your existing `trained_models/efficientnetb0_colon.h5` file using the cell below or the Colab files panel.

In [ ]:
from google.colab import files
import os

if not os.path.exists('efficientnetb0_colon.h5'):
    print('Please upload your efficientnetb0_colon.h5 model:')
    uploaded = files.upload()
else:
    print('Model efficientnetb0_colon.h5 already exists.')

### Step 3: Start Inference Server and Public Tunnel
Enter your free [ngrok](https://ngrok.com) auth token below to expose a public HTTPS endpoint.

In [ ]:
import os
import time
import base64
import numpy as np
import cv2
from io import BytesIO
from PIL import Image
from flask import Flask, request, jsonify
import tensorflow as tf
from pyngrok import ngrok
import threading

NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"  # <-- Paste your free ngrok token here
MODEL_PATH = 'efficientnetb0_colon.h5'

print('[*] Loading EfficientNetB0 model...')
model = tf.keras.models.load_model(MODEL_PATH)
_ = model(tf.zeros((1, 224, 224, 3)))
print('[✓] Model loaded successfully!')

app = Flask(__name__)
CLASS_LABELS = {0: 'colon_aca', 1: 'colon_n'}
CLASS_DISPLAY_NAMES = {'colon_aca': 'Colon Adenocarcinoma', 'colon_n': 'Colon Normal Tissue'}

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'healthy', 'tf_version': tf.__version__})

@app.route('/predict', methods=['POST'])
def predict():
    if 'file' not in request.files and not request.data:
        return jsonify({'error': 'No file provided'}), 400
    image_bytes = request.files['file'].read() if 'file' in request.files else request.data
    
    # Preprocess
    pil_img = Image.open(BytesIO(image_bytes)).convert('RGB')
    orig_np = np.array(pil_img)
    resized = cv2.resize(orig_np, (224, 224))
    tensor = np.expand_dims(resized.astype(np.float32), axis=0)
    
    # Predict
    start_time = time.perf_counter()
    predictions = model.predict(tensor, verbose=0)
    inference_time = (time.perf_counter() - start_time) * 1000
    
    pred_idx = int(np.argmax(predictions[0]))
    confidence = float(predictions[0][pred_idx])
    predicted_class = CLASS_LABELS[pred_idx]
    conf_level = 'high' if confidence >= 0.85 else ('medium' if confidence >= 0.60 else 'low')
    
    # Grad-CAM on top_conv
    base_model = model.layers[0] if isinstance(model.layers[0], tf.keras.Model) else None
    if base_model:
        target_layer = base_model.get_layer('top_conv')
        grad_model = tf.keras.models.Model(inputs=base_model.inputs, outputs=[target_layer.output, base_model.output])
        with tf.GradientTape() as tape:
            conv_out, base_out = grad_model(tensor)
            x = base_out
            for layer in model.layers[1:]:
                x = layer(x)
            loss = x[:, pred_idx]
        grads = tape.gradient(loss, conv_out)
        pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
        heatmap = tf.squeeze(conv_out[0] @ pooled[..., tf.newaxis])
        heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
        heatmap_np = heatmap.numpy()
    else:
        heatmap_np = np.ones((7, 7), dtype=np.float32)
        
    heatmap_resized = cv2.resize(heatmap_np, (orig_np.shape[1], orig_np.shape[0]))
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_rgb = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    overlay = cv2.addWeighted(orig_np, 0.6, heatmap_rgb, 0.4, 0)
    
    _, buffer = cv2.imencode('.jpg', cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
    heatmap_b64 = base64.b64encode(buffer).decode('utf-8')
    
    return jsonify({
        'predicted_class': predicted_class,
        'display_name': CLASS_DISPLAY_NAMES[predicted_class],
        'confidence': round(confidence, 4),
        'confidence_level': conf_level,
        'all_probabilities': {CLASS_LABELS[i]: round(float(p), 4) for i, p in enumerate(predictions[0])},
        'inference_time_ms': round(inference_time, 2),
        'model_version': '1.0.0',
        'heatmap_base64': heatmap_b64
    })

# Set ngrok token and start tunnel
if NGROK_AUTH_TOKEN and NGROK_AUTH_TOKEN != 'YOUR_NGROK_AUTH_TOKEN_HERE':
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    tunnel = ngrok.connect(5000)
    print(f'\n======================================================')
    print(f'  PUBLIC URL: {tunnel.public_url}/predict')
    print(f'  Copy this URL to your Windows .env:')
    print(f'  REMOTE_INFERENCE_URL={tunnel.public_url}/predict')
    print(f'======================================================\n')

app.run(port=5000)